# Advanced CTF Analysis: Ping Displays and Reference-Location Alignment

## Part of the open_dvm Toolbox

This tutorial builds on `07_ctf_analysis.ipynb` to reconstruct spatial tuning on homogeneous ("ping") displays — main-task trials containing neither a target nor a distractor — and aligns each subject's reconstruction to a subject-specific reference location (their individually assigned high-probability distractor location) using the `special_loc` parameter.

## Learning Objectives

After completing this tutorial, you will:

- **Identify homogeneous ("ping") displays** — Select main-task trials free of target/distractor confounds
- **Run cross-task CTF training with a held-out test set** — Train on localizer trials, test on ping trials from the main task
- **Align reconstructions to a subject-specific reference location** — Use `special_loc` to recenter each subject's tuning function to their own high-probability distractor location
- **Aggregate results across subjects** — Combine per-subject CTF reconstructions into a group-level result

**Prerequisites:** This tutorial assumes familiarity with `07_ctf_analysis.ipynb`, in particular the `cnds` cross-condition syntax for training/testing on different conditions.

## Overview

### Key Steps
1. **Ping displays**: Identify homogeneous main-task trials (no target, no distractor)
2. **Cross-task CTF with `special_loc`**: Train on localizer trials, test on ping trials, aligned per subject to their high-probability distractor location — run across all subjects
3. **Group-level aggregation**: Load and visualize the reconstruction averaged across subjects

## Section 1: Setup and Configuration

### 1.1 Import Required Libraries

In [1]:
%matplotlib inline

import sys, os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec

warnings.filterwarnings('ignore')
sys.path.insert(0, '/Users/dvm/Documents/DvM')

from open_dvm.analysis import CTF
from open_dvm.support.FolderStructure import FolderStructure
from open_dvm.visualization.plot import plot_ctf_timecourse

print("✓ Imports successful")

✓ Imports successful


### 1.2 Configuration

In [ ]:
# ============================================
# Configuration: Change these for your data
# ============================================
project_folder = '/Users/dvm/Library/CloudStorage/OneDrive-VrijeUniversiteitAmsterdam/projects/openDvM'
os.chdir(project_folder)

# Eye-tracking quality control
eye_dict = {
    'use_tracker': True,  # Enable eye-tracking exclusion
    'window_oi': (0, 0.3),  # Window: 0-300 ms post-stimulus
    'angle_thresh': 1,  # Threshold: 1 degree visual angle
    'viewing_dist': 70,  # Viewing distance (cm)
    'screen_res': (1920, 1080),  # Screen resolution (pixels)
    'screen_h': 29,  # Screen height (cm)
    'drift_correct': (-0.2, 0)  # Drift correction window
}

---

## Section 2: Cross-Task CTF on Homogeneous ("Ping") Displays

**Question**: Is there measurable spatial tuning on "ping" displays — main-task trials with neither a target nor a distractor present — when reconstructions are aligned to each subject's own high-probability distractor location?

**Approach**: For each subject, train a CTF on localizer trials (which sample all 8 position bins) and test on homogeneous main-task trials, excluding heterogeneous (target- or distractor-present) trials via `excl_factor`. Because each subject was individually assigned a different high-probability distractor location, we use `special_loc=int(df_sj.high_prob_dist.iloc[0])` to recenter every subject's test-trial labels to that location before reconstruction. This is what lets us average tuning functions across subjects in a shared reference frame, even though the true high-probability location differs from subject to subject.

In [ ]:
# Identify homogeneous ("ping") displays and run cross-task CTF for all subjects
print('\nRunning cross-task CTF with special_loc alignment for all subjects...')
for subject_id in range(1, 8):
    try:
        # Load data for this subject
        df_sj, epochs_sj = FolderStructure().load_processed_epochs(
            subject_id, 'ses_01_main', 'main', eye_dict
        )

        # Flag main-task trials with neither target nor distractor present ('homogeneous')
        df_sj['display_type'] = 'homogeneous'
        df_sj.loc[((df_sj.target_presence != 'absent') |
                  (df_sj.distractor_presence != 'absent')) &
                  (df_sj.block_type == 'main'),
                  'display_type'] = 'heterogeneous'

        ctf_o = CTF(
            sj=subject_id, epochs=epochs_sj, df=df_sj, to_decode='img_loc',  # Reconstruct tuning for spatial position (img_loc)
            nr_bins=8,             # 8 discrete position bins (matches 8 possible stimulus locations)
            nr_chans=8,            # 8 hypothetical spatial channels underlying the basis set
            elec_oi='all',         # Use all electrodes
            filter=8,              # Lowpass filter at 8 Hz for broadband voltage reconstruction
            avg_ch=True,           # Average tuning across bins (not per-bin reconstruction)
            baseline=(-0.2, 0),    # Baseline correction: -200 to 0 ms
            downsample=128,        # Downsample to 128 Hz
            ctf_param='von_mises', # Extract slopes + von Mises fit (amplitude, baseline, concentration, mean location)
        )

        # Train on localizer, test on homogeneous ping trials, aligned to this
        # subject's own high-probability distractor location
        ctf_o.spatial_ctf(
            pos_labels='all',                                 # Use all position bins
            cnds=dict(block_type=[['localizer'], ['main']]),  # Train on localizer, test on main
            window_oi=(-0.2, 0.5),                            # Analysis window: -200 to 500 ms
            freqs='broadband',                                # Broadband (unfiltered) voltage reconstruction
            excl_factor=dict(display_type=['heterogeneous']), # Restrict test set to homogeneous ping trials
            special_loc=int(df_sj.high_prob_dist.iloc[0]),    # Recenter test labels to this subject's reference location
            f_name='ctf_ping'                                 # Analysis identifier for saved output
        )

        print(f'  ✓ Subject {subject_id} complete')
    except Exception as e:
        print(f'  ✗ Subject {subject_id} failed: {str(e)}')

print('\n✓ All subjects processed')

---

## Section 3: Aggregating and Visualizing the Group-Level Reconstruction

**Question**: Averaged across all subjects, does the ping-display reconstruction show reliable tuning around the (now-shared) reference location?

**Approach**: Load each subject's saved CTF parameters (`f_name='ctf_ping'`) and plot the group-averaged voltage amplitude timecourse with a t-test against baseline.

In [ ]:
# Load per-subject ping CTF parameters saved in the previous step
ctfs_ping = FolderStructure().read_ctfs(
    ctf_folder_path=['img_loc'], output_type='param', ctf_name='ctf_ping', sjs='all'
)

In [ ]:
# Visualize group-level ping reconstruction (voltage amplitude, t-test vs. baseline)
fig = plt.figure(figsize=(12, 5))
plot_ctf_timecourse(ctfs_ping, output='voltage_amps', timecourse='1d', stats='ttest', colors='red')
display(fig)

print('✓ Group-level ping reconstruction complete')

---

## Section 4: Summary and Next Steps

You've now used the CTF framework to answer a genuinely different kind of question than the within-task tuning in `07_ctf_analysis.ipynb`:

1. ✅ **Homogeneous displays**: Isolated "ping" trials free of target/distractor confounds
2. ✅ **`special_loc` alignment**: Recentered per-subject reconstructions to a subject-specific reference location, enabling group averaging despite each subject having a different true high-probability location
3. ✅ **Group-level reconstruction**: Aggregated and statistically tested spatial tuning across the full subject group

This concludes the CTF tutorial series. Together, `07_ctf_analysis.ipynb` and this notebook cover the core spatial-encoding-model workflow in `open_dvm`: computing tuning functions, comparing frequency bands and power types, testing cross-task generalization, and aligning reconstructions to subject-specific reference locations for group-level analysis.